# Ollama Cloud — run huge models with no GPU

Ollama Cloud runs models on **Ollama's servers**, so you can use models far bigger than
your hardware holds — `gpt-oss:120b`, big Llamas, etc. — with **no local GPU**. There's a
**free tier** ($0, light usage).

Contrast with what you self-host on this cluster: the L40S tops out around a 70B. Cloud
picks up where your hardware stops.

**Security:** this notebook asks for your API key at runtime (hidden input). It is **not**
stored in the notebook, on disk, or in git. Get a free key at [ollama.com](https://ollama.com) → API keys → *Add API Key*.

In [ ]:
!pip install -q ollama requests

In [ ]:
import os, getpass

# Prompt for the key (hidden). Not saved anywhere.
if not os.environ.get("OLLAMA_API_KEY"):
    os.environ["OLLAMA_API_KEY"] = getpass.getpass("Ollama API key: ")

OLLAMA_CLOUD = "https://ollama.com"
AUTH = {"Authorization": "Bearer " + os.environ["OLLAMA_API_KEY"]}
print("Key loaded (", len(os.environ["OLLAMA_API_KEY"]), "chars ).")

## What cloud models can I call?

In [ ]:
import requests

r = requests.get(f"{OLLAMA_CLOUD}/api/tags", headers=AUTH, timeout=30)
r.raise_for_status()
models = [m["name"] for m in r.json().get("models", [])]
print("Cloud models available to your account:")
for m in models:
    print(" •", m)

## Chat with a big model (no GPU on your side)

Pick one from the list above. `gpt-oss:120b` is a good "far bigger than my L40S" example.
Streaming so you watch it think in real time.

In [ ]:
from ollama import Client

client = Client(host=OLLAMA_CLOUD, headers=AUTH)

MODEL = "gpt-oss:120b"   # change to any model from the list above

messages = [{"role": "user", "content": "In 3 sentences, what makes a 120B model different from a 3B one?"}]
for part in client.chat(MODEL, messages=messages, stream=True):
    print(part["message"]["content"], end="", flush=True)
print()

## The point

That `120b` answer came from a model **way too big for your L40S (44 GB)** — it ran on
Ollama's servers, on the **free tier**, with **no GPU node** and **no 40 GB download** on
your side.

Your platform now spans the full range:

| Size | Where it runs |
|------|---------------|
| 3B | self-host on **CPU** (`ollama.ollama`) |
| 70B | self-host on **GPU / L40S** (`ollama-gpu.ollama`) |
| 120B / 405B | **Ollama Cloud** (this notebook) |

Self-host to learn the infra and keep data private; cloud for the giants you can't hold.

*Part of the [AI-ML Unified Playground](https://github.com/suvmaha/ai-ml-unified-playground-platform) — see `/ollama.html` for the local-vs-cloud hub.*